# Data Mining Assignment: Apache Spark Data Lake Preprocessing

This notebook demonstrates a simple Data Lake structure in Google Colab. The raw data remains unchanged in the Bronze layer, basic preprocessing is applied using Apache Spark, and the processed data is saved in the Silver layer.


## 1. Data Lake Structure
Creating the `data_lake/bronze/` and `data_lake/silver/` directories.


In [13]:
import os

# Create Bronze and Silver layer directories
os.makedirs("data_lake/bronze", exist_ok=True)
os.makedirs("data_lake/silver/preprocessed_sales", exist_ok=True)

print("Data Lake directories created successfully.")


Data Lake directories created successfully.


## 2. Load and Inspect with Spark
Installing PySpark, creating a SparkSession, and loading the Bronze dataset.


In [14]:
!pip install pyspark -q

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, count, when, isnan, trim, lower, initcap, to_date, round
from pyspark.sql.types import IntegerType, DoubleType

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("DataLakePreprocessing") \
    .getOrCreate()

print("Spark Session created successfully.")


Spark Session created successfully.


Assuming you have uploaded `sales.csv` to Colab's default directory, let's copy it to the bronze layer first.


In [15]:
import shutil

# Make sure to upload 'sales.csv' to the Colab session storage before running this!
if os.path.exists("sales.csv"):
    shutil.copy("sales.csv", "data_lake/bronze/sales.csv")
    print("sales.csv copied to Bronze layer.")
else:
    print("Please upload sales.csv to the Colab runtime! Then run this cell again.")


sales.csv copied to Bronze layer.


In [16]:
# Read the raw data from the Bronze layer
bronze_path = "data_lake/bronze/sales.csv"
df = spark.read.csv(bronze_path, header=True, inferSchema=True)

# Display schema
df.printSchema()

# Display a few rows
df.show(5)

# Total record count
total_records = df.count()
print(f"Total records in Bronze: {total_records}")


root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id| customer_name|      product|category|quantity|unit_price|discount_percent|payment_method|      city|         state|order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------

### Inspecting Data
Identifying duplicates, null values, text-format inconsistencies, invalid numerical values, and inconsistent dates.


In [17]:
# 1. Identify duplicates
duplicates_count = df.count() - df.dropDuplicates().count()
print(f"Number of exact duplicate rows: {duplicates_count}")

# 2. Identify Null/Missing Values safely
from pyspark.sql.types import DoubleType, FloatType

null_counts = []
for f in df.schema.fields:
    c = f.name
    # Only use isnan() on Double or Float types to avoid CAST_INVALID_INPUT errors
    if isinstance(f.dataType, (DoubleType, FloatType)):
        null_counts.append(count(when(isnan(col(c)) | col(c).isNull(), c)).alias(c))
    else:
        null_counts.append(count(when(col(c).isNull(), c)).alias(c))

print("Null values per column:")
df.select(null_counts).show()


Number of exact duplicate rows: 10
Null values per column:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



## 3. Basic Preprocessing - Silver Layer
Applying standard data cleaning operations as requested.


In [18]:
# 1. Remove exact duplicate rows
df_cleaned = df.dropDuplicates()

# Get text and numeric columns dynamically to handle the dataset generically
text_columns = [f.name for f in df_cleaned.schema.fields if f.dataType.typeName() == 'string']

for c in text_columns:
    # 2. Trim extra spaces & 3. Standardize capitalization
    df_cleaned = df_cleaned.withColumn(c, trim(initcap(col(c))))

# Define typical target columns based on assignment description
quantity_cols = [c for c in df_cleaned.columns if "quantity" in c.lower() or c.lower() == "qty"]
price_cols = [c for c in df_cleaned.columns if "price" in c.lower()]
discount_cols = [c for c in df_cleaned.columns if "discount" in c.lower()]
date_cols = [c for c in df_cleaned.columns if "date" in c.lower()]

# 4. Basic validation to numerical fields
# Filter invalid quantities (e.g., negative or zero)
for c in quantity_cols:
    df_cleaned = df_cleaned.filter((col(c) > 0) | col(c).isNull())

# Filter invalid prices (negative)
for c in price_cols:
    df_cleaned = df_cleaned.filter((col(c) >= 0) | col(c).isNull())

# Validate discount percentage (between 0 and 1 or 0 and 100)
for c in discount_cols:
    df_cleaned = df_cleaned.filter((col(c) >= 0) & (col(c) <= 100) | col(c).isNull())

# 5. Convert valid dates to one consistent format
for c in date_cols:
    # Spark to_date defaults to yyyy-MM-dd
    df_cleaned = df_cleaned.withColumn(c, to_date(col(c)))

# 2b. Handle missing values
# For numeric columns, fill nulls with 0 or a reasonable value.
# For simplicity, filling numeric nulls with 0, and string nulls with 'Unknown'
numeric_columns = [f.name for f in df_cleaned.schema.fields if f.dataType.typeName() in ['integer', 'double', 'long']]

if numeric_columns:
    df_cleaned = df_cleaned.na.fill(0, subset=numeric_columns)
if text_columns:
    df_cleaned = df_cleaned.na.fill("Unknown", subset=text_columns)


### Save to Silver Layer
Saving the data as a single file and renaming it to `sales_silver.csv` inside the `silver/` directory to match the submission requirements.


In [19]:
from pyspark.sql.functions import coalesce

# 1. Remove exact duplicate rows
df_cleaned = df.dropDuplicates()

# Get text and numeric columns dynamically to handle the dataset generically
text_columns = [f.name for f in df_cleaned.schema.fields if f.dataType.typeName() == 'string']

for c in text_columns:
    # 2. Trim extra spaces & 3. Standardize capitalization
    df_cleaned = df_cleaned.withColumn(c, trim(initcap(col(c))))

# Define typical target columns based on assignment description
quantity_cols = [c for c in df_cleaned.columns if "quantity" in c.lower() or c.lower() == "qty"]
price_cols = [c for c in df_cleaned.columns if "price" in c.lower()]
discount_cols = [c for c in df_cleaned.columns if "discount" in c.lower()]
date_cols = [c for c in df_cleaned.columns if "date" in c.lower()]

# 4. Basic validation to numerical fields
# Filter invalid quantities (e.g., negative or zero)
for c in quantity_cols:
    df_cleaned = df_cleaned.filter((col(c) > 0) | col(c).isNull())

# Filter invalid prices (negative)
for c in price_cols:
    df_cleaned = df_cleaned.filter((col(c) >= 0) | col(c).isNull())

# Validate discount percentage (between 0 and 1 or 0 and 100)
for c in discount_cols:
    df_cleaned = df_cleaned.filter((col(c) >= 0) & (col(c) <= 100) | col(c).isNull())

# 5. Convert valid dates to one consistent format and handle invalid ones gracefully
# Turn off strict ANSI mode so invalid dates like 31/02/2026 become NULL instead of crashing the program
spark.conf.set("spark.sql.ansi.enabled", "false")

for c in date_cols:
    # Try different common formats. Invalid dates will become NULL.
    df_cleaned = df_cleaned.withColumn(c, coalesce(
        to_date(col(c), "dd/MM/yyyy"),
        to_date(col(c), "MM/dd/yyyy"),
        to_date(col(c), "yyyy-MM-dd")
    ))

# 2b. Handle missing values
# For numeric columns, fill nulls with 0
# For string columns, fill nulls with 'Unknown'
numeric_columns = [f.name for f in df_cleaned.schema.fields if f.dataType.typeName() in ['integer', 'double', 'long']]

if numeric_columns:
    df_cleaned = df_cleaned.na.fill(0, subset=numeric_columns)
if text_columns:
    df_cleaned = df_cleaned.na.fill("Unknown", subset=text_columns)


## 4. Verification
Show row counts, read Silver data back, and display.


In [21]:
import glob
import shutil
import os

silver_dir = "data_lake/silver/preprocessed_sales"
os.makedirs("data_lake/silver", exist_ok=True)

# Coalesce to 1 partition to ensure a single output CSV file
df_cleaned.coalesce(1).write.mode("overwrite").csv(silver_dir, header=True)

# Find the part file generated by Spark and rename it to 'sales_silver.csv'
part_files = glob.glob(f"{silver_dir}/*.csv")
if part_files:
    silver_output_file = "data_lake/silver/sales_silver.csv"
    shutil.copy(part_files[0], silver_output_file)
    print(f"Preprocessed data correctly saved and renamed to: {silver_output_file}")
else:
    print(f"Error: No CSV files found in {silver_dir}")


Preprocessed data correctly saved and renamed to: data_lake/silver/sales_silver.csv
